In [7]:
#Ishaan
#Labels
import math
import pandas as pd
import numpy as np
def soft_labels_generation(ratings):
    soft_list = []
    for i in ratings: 
        temp_list = [int(x) for x in i.split(";")]
        n = len(temp_list)
        soft_list.append([temp_list.count(1)/n, temp_list.count(2)/n,temp_list.count(3)/n,temp_list.count(4)/n,temp_list.count(5)/n])
    return np.array(soft_list) # shape (num_nodules, 5)

def hard_labels_generation(ratings):
    hard_list = []
    for i in ratings:
        temp_list = sorted(int(x) for x in i.split(";")) # median, not mean: mean gives floats (regression); median keeps 5 classes
        n = len(temp_list)
        median = temp_list[n//2] if n % 2 else (temp_list[n//2 - 1] + temp_list[n//2]) / 2
        hard_list.append(math.floor(median + 0.5))     #not round(): banker's rounding
    return np.array(hard_list) # shape (num_nodules,), values 1..5
        
labels_bb_a = "data/labels_bb_a.csv"
df = pd.read_csv(labels_bb_a)
nodules = df.drop_duplicates(["patient_id", "series_instance_uid", "merged_nodule_id"]).copy()
nodules = nodules[nodules["num_readers"] >= 2].reset_index(drop=True)
soft_labels = soft_labels_generation(nodules["ratings"])
hard_labels = hard_labels_generation(nodules["ratings"])
# 4;5;5;5 ratings = [4, 5, 5, 5]  →  hard = 5,  soft = [0, 0, 0, 0.25, 0.75]
# ratings = [1, 2, 4, 5]  →  hard = 3,  soft = [0.25, 0.25, 0, 0.25, 0.25]


[[0.   0.   0.   0.25 0.75]
 [0.   0.   0.   0.5  0.5 ]
 [0.   0.   0.25 0.25 0.5 ]
 ...
 [0.   0.   0.25 0.   0.75]
 [0.   0.   0.5  0.5  0.  ]
 [0.   1.   0.   0.   0.  ]]


In [10]:
#Ishaan
#Labels
import math
import numpy as np
import pandas as pd

LABELS_BB_A = "data/labels_bb_a.csv"
OUT_PATH    = "data/nodules_labels.csv"
MIN_READERS = 2


def parse_ratings(s):
    # "4;5;5;5" -> [4, 5, 5, 5]   (str() in case a single rating was read as a number)
    return [int(x) for x in str(s).split(";")]


def soft_labels_generation(ratings):
    soft_list = []
    for s in ratings:
        r = parse_ratings(s)
        n = len(r)
        soft_list.append([r.count(k) / n for k in (1, 2, 3, 4, 5)])
    return np.array(soft_list)            # shape (num_nodules, 5)


def hard_labels_generation(ratings):
    # median, not mean: mean gives floats (regression); median keeps 5 classes
    hard_list = []
    for s in ratings:
        r = sorted(parse_ratings(s))
        n = len(r)
        median = r[n//2] if n % 2 else (r[n//2 - 1] + r[n//2]) / 2
        hard_list.append(math.floor(median + 0.5))    # not round(): banker's rounding
    return np.array(hard_list)            # shape (num_nodules,), values 1..5


# ---- 1. load ----
df = pd.read_csv(LABELS_BB_A)
if "ratings" not in df.columns:
    raise SystemExit(
        f"{LABELS_BB_A} has no 'ratings' column: it's the old file. "
        "Rerun xml_preprocessing_bb_a.py to regenerate it first.\n"
        f"Columns found: {list(df.columns)}")

# ---- 2. one row per nodule (the CSV has one row per slice) ----
# NOT drop_duplicates: that keeps whichever slice came first, which is the nodule's
# top EDGE -- narrowest box, drawn by fewest readers. Anchor on the widest slice
# (the nodule's equator) so the centre and diameter describe the actual nodule.
key = ["patient_id", "series_instance_uid", "merged_nodule_id"]
df["box"] = df[["width", "height"]].max(axis=1)

nodules = df.loc[df.groupby(key)["box"].idxmax()].copy()
nodules = nodules.rename(columns={"image_sop_id": "anchor_sop_id",
                                  "z-slice":      "anchor_z",
                                  "box":          "diameter_px"})
nodules = nodules.merge(df.groupby(key).size().rename("n_slices").reset_index(), on=key)

# ---- 3. same filter for hard AND soft ----
nodules = nodules[nodules["num_readers"] >= MIN_READERS].reset_index(drop=True)

# ---- 4. labels, built from NODULES, not df ----
hard = hard_labels_generation(nodules["ratings"])
soft = soft_labels_generation(nodules["ratings"])

# ---- 5. attach as columns so they can never get out of order ----
nodules["hard"] = hard
for k in range(5):
    nodules[f"soft_{k+1}"] = soft[:, k]

# how far apart the readers were: 0 = unanimous, 4 = one said 1 and another said 5.
# Fixed now, before seeing any model output, so "high disagreement" isn't chosen to fit results.
nodules["spread"] = nodules["ratings"].apply(lambda s: max(parse_ratings(s)) - min(parse_ratings(s)))

# keep nodule-level columns only; per-slice boxes stay in labels_bb_a.csv
keep = (key + ["num_readers", "ratings", "hard"] + [f"soft_{k}" for k in range(1, 6)]
        + ["spread", "anchor_sop_id", "anchor_z", "x_centre", "y_centre",
           "diameter_px", "n_slices"])
nodules = nodules[keep]
nodules.to_csv(OUT_PATH, index=False)

# ---- 6. checkpoints ----
print(f"{len(df)} slice rows -> {len(nodules)} nodules (num_readers >= {MIN_READERS})")
assert len(nodules) == len(soft) == len(hard)
assert np.allclose(soft.sum(axis=1), 1.0), "a soft label doesn't sum to 1"
assert set(np.unique(hard)) <= {1, 2, 3, 4, 5}, "hard label outside 1..5"
assert nodules["num_readers"].max() <= 4, "more than 4 readers: rerun the parser"
print("hard label counts:\n", nodules["hard"].value_counts().sort_index())
print("spread counts:\n", nodules["spread"].value_counts().sort_index())
print(nodules[nodules["patient_id"] == "LIDC-IDRI-0001"].to_string(index=False))
print(f"saved {OUT_PATH}")


16172 slice rows -> 1885 nodules (num_readers >= 2)
hard label counts:
 hard
1    206
2    288
3    898
4    316
5    177
Name: count, dtype: int64
spread counts:
 spread
0    409
1    718
2    552
3    179
4     27
Name: count, dtype: int64
    patient_id                                              series_instance_uid  merged_nodule_id  num_readers ratings  hard  soft_1  soft_2  soft_3  soft_4  soft_5  spread                                                    anchor_sop_id  anchor_z  x_centre  y_centre  diameter_px  n_slices
LIDC-IDRI-0001 1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636438705059720603192                 1            4 5;5;5;4     5     0.0     0.0     0.0    0.25    0.75       1 1.3.6.1.4.1.14519.5.2.1.6279.6001.824843590991776411530080688091    -117.5   316.375    363.25         43.5         9
saved data/nodules_labels.csv
